In [ ]:
import numpy

import matplotlib.pyplot as plt
plt.style.use('mystyle.mplstyle')

import sbruceana

In [ ]:
PATH_TO_SBRUCE = "/Users/triozzi/Analysis/nuedis/checks/data/"

FILE_NUE_CV = "CNAF_NuE_1eNp0pi_NuMI_2100.root"

BINNING = numpy.array([0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1., 1.1, 1.2, 1.3, 1.4, 1.5, 2, 3])

In [ ]:
# open sbruce tree
file = sbruceana.io.get_root_file(f'{PATH_TO_SBRUCE}{FILE_NUE_CV}')

# get per-syst covariance matrices
cov_results, cv, bin_centers = sbruceana.systs.build_covariance_matrix(
  file, 
  BINNING, 
  'recoE'
)

# get fractional systematics
frac_by_group, frac_total = sbruceana.systs.get_fractional_uncertainty(
  cov_results, 
  cv
)

#### Covariance matrix

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

ax = sbruceana.plotting.plot_covariance_matrix(ax, cov_results, cv, bin_centers)

#### Plotting with full systematic uncertainty

In [ ]:
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_NUE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_NUE_CV}")

In [ ]:
fig, ax = plt.subplots(figsize=(4,3), layout='constrained')

var = "recoE"

# width = 0.2; bins = numpy.arange(0.2, 3+width, width)
bins = BINNING

# CV
ax = sbruceana.plotting.plot_var_by_category(ax, df, bins, var, False, hatch='///')

# systematic uncertainty
ax = sbruceana.plotting.plot_error_band(ax, cov_results, cv, bin_centers, BINNING, groups=['fluxsyst'], fc='gray', ec='gray', hatch='///', label='flux')
ax = sbruceana.plotting.plot_error_band(ax, cov_results, cv, bin_centers, BINNING, groups=['fluxsyst', 'OtherXSec', 'QE-MEC', 'FSI', 'NonRESBG'], fc='gray', ec='gray', hatch='\\\\\\', label='flux+x-sec')

# gfx
ax.set(
  title  = f'CV',
  xlabel = f'{var} [GeV]',
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(title=f'{pot:.2e} POT'); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig("plots/CNAF_NuE_1eNp0pi_NuMI.png", dpi=300)

#### Fractional systematic uncertainty

In [ ]:
fig, ax = plt.subplots(figsize=(4,3), layout='constrained')

for tag, frac in frac_by_group.items():
    ax.step(BINNING, numpy.append(frac, frac[-1]), where="post", label=tag)
ax.step(BINNING, numpy.append(frac_total, frac_total[-1]),
        where="post", label="Total", color="black", linestyle="-", linewidth=2)

ax.set(
  ylim = (0, 0.4),
  xlim = (BINNING[0], BINNING[-1]),
  ylabel = 'Fractional uncertainty',
  xlabel = 'recoE [GeV]',
)

ax.legend()